# MTL analysis from the importance matrix (Section 5)

TSsBN turns a trained model's scalers into a **task-filter importance matrix**
`I = sigmoid(gamma)`, shape `[tasks, filters]` — exactly what `tssnorm.get_importance(model)`
returns. Because it is read straight from the trained parameters, the whole Section 5 analysis
is *free*: no extra forward passes, no ablation.

`tssnorm.analysis` implements that toolkit as pure functions of `I`: capacity and its
shared/independent decomposition, task similarity and clustering, filter specialization, and
cross-run stability.

The run below uses the importance matrix of a trained TSsBN model — the NYUv2 three-task
DeepLabV3 + ResNet-50 — to show each function on real scalers rather than toy numbers.

Importance matrices are not committed. Save one from your own model with
`numpy.save(path, get_importance(model))` and point the loader at it; a many-task model
(e.g. 40-task CelebA) additionally makes the clustering in 5.2 rich.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from tssnorm.analysis import capacity, capacity_decomposition, task_similarity, specialized_filters

# A [tasks, filters] importance matrix from a trained model: I = get_importance(model).
MATRIX = Path("models/nyuv2_tssbn.npy")
if not MATRIX.exists():
    raise FileNotFoundError(
        f"{MATRIX} not found. Save a trained model's importance matrix with "
        "numpy.save(path, get_importance(model)) and point MATRIX at it — see the README."
    )

I = np.load(MATRIX)
TASKS = ["segmentation", "depth", "normals"]
print(f"importance matrix {I.shape} = [tasks, filters]  tasks={TASKS}")

## Capacity allocation (5.1, Fig. 3)

`capacity(I)` is each task's total capacity `C_t = mean_i I[t,i]`.
`capacity_decomposition(I)` splits it into a **shared** part (the projection of the task's
importance vector onto the span of the other tasks) and an **independent** part (the residual).
The two are *orthogonal*, so they combine in quadrature: `shared^2 + independent^2 = C_t^2`.
That is why the plot stacks the **squared** components — they are what adds up to `C_t^2`, and
it is the quantity Figure 3 shows. Most capacity comes out shared, with no single task
dominating.

In [ ]:
C = capacity(I)
shared, indep = capacity_decomposition(I)

fig, ax = plt.subplots(figsize=(5, 4))
x = np.arange(len(TASKS))
ax.bar(x, shared**2, label="shared", color="#4C72B0")
ax.bar(x, indep**2, bottom=shared**2, label="independent", color="#DD8452")
ax.set_xticks(x); ax.set_xticklabels(TASKS)
ax.set_ylabel("capacity$^2$"); ax.set_title("Per-task capacity: shared vs independent (Fig. 3)")
ax.legend(); fig.tight_layout(); plt.show()

# the shared fraction of capacity is likewise a ratio of squares
for t, c, s in zip(TASKS, C, shared):
    print(f"  {t:13s} capacity {c:.3f}   shared {s**2 / c**2:.0%}")

## Task relationships (5.2)

`task_similarity(I)` is the pairwise cosine similarity between task importance vectors — 1 for
identical filter usage, 0 for orthogonal. (With many tasks, `task_clusters(I)` turns this into
the hierarchical clustering of 5.2 / Table 5; with three tasks we just read the matrix.)

In [ ]:
S = task_similarity(I)
fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(S, cmap="magma", vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(TASKS, rotation=30, ha="right")
ax.set_yticks(range(3)); ax.set_yticklabels(TASKS)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{S[i,j]:.2f}", ha="center", va="center",
                color="white" if S[i,j] < 0.6 else "black", fontsize=9)
ax.set_title("Task similarity"); fig.tight_layout(); plt.show()

## Filter specialization (5.3)

`specialized_filters(I, tau)` marks a filter as specialized for a task when that task claims
more than `tau` of the filter's total importance across tasks. Most filters are shared; a
minority specialize to one task.

In [ ]:
spec = specialized_filters(I, tau=0.5)
print(f"{spec.any(axis=0).mean():.1%} of filters are specialized to a single task (tau=0.5)")
for t, n in zip(TASKS, spec.sum(axis=1)):
    print(f"  {t:13s} {n:5d} specialized filters")

## Notes

- **Real TSsBN model.** The matrix is read from a trained NYUv2 TSsBN model (Table 2). The same
  functions apply to any model via `I = get_importance(model)`; the readout is free for TSsBN.
- **Many tasks.** The task-relationship clustering of 5.2 (Table 5) and the cross-run stability
  (`similarity_stability`) are best shown with many tasks — save a 40-task CelebA model's
  importance into `models/` to reproduce those.